In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline 
import pandas as pd

import scipy.signal

from astropy.io import ascii
import exoplanet as xo

import pymc3 as pm
import pymc3_ext as pmx
import lightkurve as lk
from astropy.timeseries import BoxLeastSquares, LombScargle
from astropy.stats import sigma_clip

In [ ]:
#Curva = ascii.read('Data_Transit_TIC427393298_Full.dat')

#Times = Curva.columns[0].astype(np.float64)
#Flux = Curva.columns[1].astype(np.float64)
curva = pd.read_csv("/home/luciam/Documents/starry/pleyades_5/TIC_346541355/Tic_346541355_s_44_corrected.csv")
Times = curva["time"]
Flux = curva["flux"]

fig = plt.figure(figsize=(12,5), dpi=300)
plt.plot(Times, Flux, ".k")
plt.ylabel("Flujo")
plt.xlabel("Tiempo")
plt.rc('font', size=16)

In [ ]:
# Eliminar NaNs
mask = np.isfinite(Times) & np.isfinite(Flux)
Times = Times[mask]
Flux = Flux[mask]

# Normalizar flujo
Flux_norm = Flux / np.median(Flux)

plt.figure(figsize=(12,5), dpi=300)
plt.plot(Times, Flux_norm, ".k", ms=1)
plt.xlabel("Tiempo [BTJD]")
plt.ylabel("Flujo normalizado")
plt.show()

In [ ]:
bls = BoxLeastSquares(Times, Flux_norm)

periods = np.linspace(0.5, 30, 20000)  # ajusta según duración del sector
durations = np.linspace(0.02, 0.3, 10) # días

power = bls.power(periods, durations)


In [ ]:
best = np.argmax(power.power)

period_1 = power.period[best]
t0_1 = power.transit_time[best]
duration_1 = power.duration[best]

print(period_1, t0_1, duration_1)

In [ ]:
phase = (Times - t0_1 + 0.5*period_1) % period_1 - 0.5*period_1

plt.figure(figsize=(6,4), dpi=300)
plt.plot(phase, Flux_norm, ".k", ms=2)
plt.xlabel("Fase [días]")
plt.ylabel("Flujo")
plt.show()


In [ ]:
model_1 = bls.model(Times, period_1, duration_1, t0_1)
Flux_residual = Flux_norm - model_1 + 1


In [ ]:
plt.figure(figsize=(12,5), dpi=300)
plt.plot(Times, Flux_residual, ".k", ms=1)
plt.xlabel("Tiempo")
plt.ylabel("Flujo residual")
plt.show()

In [ ]:
in_transit = np.abs(phase) < duration_1/2
depth = 1 - np.median(Flux_residual[in_transit])

print("Profundidad:", depth)


In [ ]:
n = np.floor((Times - t0_1) / period_1).astype(int)
t_mid = t0_1 + n * period_1

plt.figure(figsize=(6,4), dpi=300)
for i in np.unique(n):
    mask = np.abs(Times - (t0_1 + i*period_1)) < 0.5
    plt.plot(Times[mask] - (t0_1 + i*period_1),
             Flux_residual[mask], '.k', alpha=0.3)

plt.xlabel("Tiempo desde tránsito [días]")
plt.ylabel("Flujo")
plt.show()


In [ ]:
in_transit = np.abs(phase) < duration_1/2
depth_res = 1 - np.median(Flux_residual[in_transit])

print("Profundidad residual:", depth_res)


In [ ]:
plt.figure(figsize=(6,4), dpi=300)

for i in np.unique(n):
    mask = np.abs(Times - (t0_1 + i*period_1)) < 0.5
    plt.plot(
        Times[mask] - (t0_1 + i*period_1),
        Flux_residual[mask],
        '.k',
        alpha=0.25
    )

# Caja del tránsito residual
plt.axvspan(
    -duration_1/2,
    duration_1/2,
    color='red',
    alpha=0.15,
    label='tránsito'
)

plt.hlines(
    1 - depth_res,
    -duration_1/2,
    duration_1/2,
    colors='red',
    linewidth=2
)

plt.xlabel("Tiempo desde tránsito [días]")
plt.ylabel("Flujo residual")
plt.legend()
plt.show()


In [ ]:

from astropy.timeseries import LombScargle
import numpy as np
import matplotlib.pyplot as plt

# Frecuencias (en 1/día)
min_period = 0.3
max_period = 30

frequency = np.linspace(
    1 / max_period,
    1 / min_period,
    50000
)

ls = LombScargle(Times, Flux_norm)
power = ls.power(frequency)

period = 1 / frequency


In [ ]:
plt.figure(figsize=(7,4), dpi=300)
plt.plot(period, power, color='k', lw=0.8)
plt.xscale("log")
plt.xlabel("Periodo [días]")
plt.ylabel("Potencia Lomb–Scargle")
plt.title("Periodograma – Curva original")
plt.gca().invert_xaxis()
plt.show()


In [ ]:
ls_res = LombScargle(Times, Flux_residual)
power_res = ls_res.power(frequency)


In [ ]:
plt.figure(figsize=(7,4), dpi=300)
plt.plot(period, power_res, color='k', lw=0.8)
plt.xscale("log")
plt.xlabel("Periodo [días]")
plt.ylabel("Potencia Lomb–Scargle")
plt.title("Periodograma – Residuos")
plt.gca().invert_xaxis()
plt.show()

In [ ]:
fig, axs = plt.subplots(
    2, 1, figsize=(7,6), dpi=300, sharex=True
)

axs[0].plot(period, power, 'k', lw=0.8)
axs[0].set_ylabel("Potencia")
axs[0].set_title("Curva original")

axs[1].plot(period, power_res, 'k', lw=0.8)
axs[1].set_ylabel("Potencia")
axs[1].set_xlabel("Periodo [días]")
axs[1].set_title("Residuos")

for ax in axs:
    ax.set_xscale("log")
    ax.invert_xaxis()

plt.tight_layout()
plt.show()


In [ ]:
# Rango de periodos
min_period = 0.2      # días (planetas cercanos)
max_period = 50       # días (rotación estelar)

frequency = np.linspace(
    1 / max_period,
    1 / min_period,
    60000
)

ls = LombScargle(Times, Flux_norm)
power = ls.power(frequency)

period = 1 / frequency

In [ ]:
plt.figure(figsize=(7,4), dpi=300)
plt.plot(period, power, color='k', lw=0.8)
plt.xscale("log")
plt.gca().invert_xaxis()
plt.xlabel("Periodo [días]")
plt.ylabel("Potencia Lomb–Scargle")
plt.title("Periodograma – Curva original")
plt.show()

In [ ]:
ls_res = LombScargle(Times, Flux_residual)
power_res = ls_res.power(frequency)


In [ ]:
plt.figure(figsize=(7,4), dpi=300)
plt.plot(period, power_res, color='k', lw=0.8)
plt.xscale("log")
plt.gca().invert_xaxis()
plt.xlabel("Periodo [días]")
plt.ylabel("Potencia Lomb–Scargle")
plt.title("Periodograma – Residuos (planetas removidos)")
plt.show()


In [ ]:
# ==========================================
# TEST 1 — ARMÓNICOS
# ==========================================
def is_harmonic(P_test, P_ref, tol=0.05):
    ratios = np.array([0.5, 1/3, 2.0, 3.0])
    return np.any(np.abs(P_test / P_ref - ratios) < tol)

harmonics = []

for P in planet_periods:
    if is_harmonic(P, P_a) or is_harmonic(P, P_b):
        harmonics.append(P)

print("Armónicos detectados:")
for P in harmonics:
    print(f"P = {P:.4f} días")


In [ ]:
# ==========================================
# TEST 2 — POTENCIA RELATIVA
# ==========================================
for P, Pow, Rpow in zip(peak_periods, peak_powers, res_powers):
    print(f"P = {P:7.3f} d | Power = {Pow:6.3f} | Residual = {Rpow:6.3f}")


In [ ]:
# ==========================================
# TEST 3 — FASE PLEGADA
# ==========================================
def phase_fold(t, P):
    return (t % P) / P

fig, ax = plt.subplots(figsize=(6,4), dpi=200)

for P in planet_periods[:3]:
    phase = phase_fold(Times, P)
    ax.scatter(phase, Flux_norm, s=3, alpha=0.4, label=f"P={P:.2f} d")

ax.set_xlabel("Fase orbital")
ax.set_ylabel("Flujo normalizado")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
for P in planet_periods[:3]:
    phase = phase_fold(Times, P)

    plt.figure(figsize=(5,3), dpi=200)
    plt.scatter(phase, Flux_norm, s=4, alpha=0.4)
    plt.title(f"Fase plegada — P = {P:.3f} d")
    plt.xlabel("Fase orbital")
    plt.ylabel("Flujo normalizado")
    plt.tight_layout()
    plt.show()


In [ ]:
from scipy.signal import savgol_filter

# ==========================================
# APLANADO DE LA CURVA DE LUZ
# ==========================================
window = 301   # impar, ajusta según cadencia
poly   = 2

trend = savgol_filter(Flux_norm, window_length=window, polyorder=poly)
Flux_flat = Flux_norm / trend

# ==========================================
# FUNCIÓN DE FASE PLEGADA
# ==========================================
def phase_fold(t, P):
    return (t % P) / P

# ==========================================
# PLOT FASE–FLUJO APLANADO
# ==========================================
fig, ax = plt.subplots(figsize=(6,4), dpi=200)

for P in planet_periods[:3]:
    phase = phase_fold(Times, P)
    ax.scatter(phase, Flux_flat, s=4, alpha=0.4, label=f"P = {P:.3f} d")

ax.set_xlabel("Fase orbital")
ax.set_ylabel("Flujo aplanado")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def phase_fold(t, P):
    return ((t % P) / P)

P_test = 0.692  # cambia luego a 0.462 y 0.346

phase = phase_fold(Times, P_test)

# Ordenar por fase (clave para ver los mínimos bien)
idx = np.argsort(phase)

plt.figure(figsize=(6,4), dpi=200)
plt.scatter(phase[idx], Flux_flat[idx], s=5, alpha=0.5)
plt.xlabel("Fase orbital")
plt.ylabel("Flujo aplanado")
plt.title(f"Plegado en P = {P_test:.3f} d")
plt.tight_layout()
plt.show()

